# Imports

In [1]:
import pandas as pd
import requests
import os
import json
from datetime import datetime

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

Configuration

In [2]:
DAILYMED_BASE = "https://dailymed.nlm.nih.gov/dailymed/services/v2"

OUTPUT_FOLDER = "dailymed_repository"

Create Output Folder

In [3]:
DAILYMED_BASE = "https://dailymed.nlm.nih.gov/dailymed/services/v2"

OUTPUT_FOLDER = "dailymed_repository"

DailyMed Client

In [4]:
class DailyMedClient:

    BASE_URL = DAILYMED_BASE

    def get_spls_by_drug_name(
        self,
        drug_name
    ):

        try:

            response = requests.get(

                f"{self.BASE_URL}/spls.json",

                params={
                    "drug_name": drug_name
                },

                timeout=60
            )

            response.raise_for_status()

            return (

                response.json()

                .get(
                    "data",
                    []
                )
            )

        except Exception:

            return []

In [5]:
client = DailyMedClient()

results = client.get_spls_by_drug_name(
    "Opdivo"
)

print(
    f"Records: {len(results)}"
)

results[:1]

Records: 2


[{'spl_version': 8,
  'published_date': 'May 28, 2026',
  'title': 'OPDIVO QVANTIG (NIVOLUMAB AND HYALURONIDASE-NVHY) INJECTION, SOLUTION [E.R. SQUIBB & SONS, L.L.C.]',
  'setid': '7f8c38fa-42f4-4387-9e8c-7a1c66855d8b'}]

Load Drug Master

In [6]:
drug_master_df = pd.read_parquet(
    "master_repository/drug_master.parquet"
)

print(
    f"Drug Master Rows: "
    f"{len(drug_master_df):,}"
)

drug_names_df = (

    drug_master_df[
        ["brand_name"]
    ]

    .dropna()

    .drop_duplicates()

    .sort_values(
        "brand_name"
    )

    .reset_index(
        drop=True
    )
)

print(
    f"Unique Brands: "
    f"{len(drug_names_df):,}"
)

Drug Master Rows: 26,137
Unique Brands: 4,953


Create Drug Universe

In [7]:
drug_names_df = (

    drug_master_df[
        ["brand_name"]
    ]

    .dropna()

    .drop_duplicates()

    .sort_values(
        "brand_name"
    )

    .reset_index(
        drop=True
    )
)

print(
    f"Unique Brands: "
    f"{len(drug_names_df):,}"
)

Unique Brands: 4,953


DailyMed Repository Builder

In [8]:
def build_dailymed_repository(
    drug_names_df
):

    client = DailyMedClient()

    rows = []

    total = len(
        drug_names_df
    )

    for index, brand_name in enumerate(

        drug_names_df[
            "brand_name"
        ],

        start=1
    ):

        if index % 100 == 0:

            print(
                f"{index:,} / {total:,}"
            )

        spls = (
            client.get_spls_by_drug_name(
                brand_name
            )
        )

        for spl in spls:

            rows.append({

                "brand_name":
                    brand_name,

                "setid":
                    spl.get(
                        "setid"
                    ),

                "title":
                    spl.get(
                        "title"
                    ),

                "published_date":
                    spl.get(
                        "published_date"
                    ),

                "version":
                    spl.get(
                        "spl_version"
                    )
            })

    return pd.DataFrame(
        rows
    )

Build Repository

In [9]:
def build_dailymed_repository(
    drug_names_df
):

    client = DailyMedClient()

    rows = []

    total = len(
        drug_names_df
    )

    for index, brand_name in enumerate(

        drug_names_df[
            "brand_name"
        ],

        start=1
    ):

        if index % 100 == 0:

            print(
                f"{index:,} / {total:,}"
            )

        spls = (
            client.get_spls_by_drug_name(
                brand_name
            )
        )

        for spl in spls:

            rows.append({

                "brand_name":
                    brand_name,

                "setid":
                    spl.get(
                        "setid"
                    ),

                "title":
                    spl.get(
                        "title"
                    ),

                "published_date":
                    spl.get(
                        "published_date"
                    ),

                "spl_version":
                    spl.get(
                        "spl_version"
                    )
            })

    return (

        pd.DataFrame(
            rows
        )

        .drop_duplicates()

        .reset_index(
            drop=True
        )
    )

Statistics

In [10]:
dailymed_statistics = {

    "rows": 0,

    "unique_brands": 0,

    "unique_setids": 0
}

print(
    "DailyMed repository has not been built yet."
)

dailymed_statistics


DailyMed repository has not been built yet.


{'rows': 0, 'unique_brands': 0, 'unique_setids': 0}

In [11]:
dailymed_repository_df = (
    build_dailymed_repository(
        drug_names_df
    )
)

print(
    f"Rows: "
    f"{len(dailymed_repository_df):,}"
)

display(
    dailymed_repository_df.head()
)

100 / 4,953
200 / 4,953
300 / 4,953
400 / 4,953
500 / 4,953
600 / 4,953
700 / 4,953
800 / 4,953
900 / 4,953
1,000 / 4,953
1,100 / 4,953
1,200 / 4,953
1,300 / 4,953
1,400 / 4,953
1,500 / 4,953
1,600 / 4,953
1,700 / 4,953
1,800 / 4,953
1,900 / 4,953
2,000 / 4,953
2,100 / 4,953
2,200 / 4,953
2,300 / 4,953
2,400 / 4,953
2,500 / 4,953
2,600 / 4,953
2,700 / 4,953
2,800 / 4,953
2,900 / 4,953
3,000 / 4,953
3,100 / 4,953
3,200 / 4,953
3,300 / 4,953
3,400 / 4,953
3,500 / 4,953
3,600 / 4,953
3,700 / 4,953
3,800 / 4,953
3,900 / 4,953
4,000 / 4,953
4,100 / 4,953
4,200 / 4,953
4,300 / 4,953
4,400 / 4,953
4,500 / 4,953
4,600 / 4,953
4,700 / 4,953
4,800 / 4,953
4,900 / 4,953
Rows: 120,884


,brand_name,setid,title,published_date,spl_version
0,1000 Roses Daily Shade Facial SPF 18,0923e090-6834-4373-80fe-21f00b8c1c17,"1000 ROSES DAILY SHADE FACIAL SPF 18 (OCTISALATE, OCTOCRYLENE, AND AVOBENZONE) LOTION [ANDALOU NATURALS]","Jun 11, 2025",7
1,1168,1183a4f5-f875-dfb0-e063-6394a90af35f,1168 (1168 BURN SPRAY) SPRAY [DYNAREX CORPORATION],"Feb 29, 2024",1
2,12-Pack Hand Sanitizer,3c9425a7-4eb9-1a63-e063-6294a90a49a4,"DARK 6-PACK HAND SANITIZER (ISOPROPYL ALCOHOL) SPRAY 12-PACK HAND SANITIZER (ISOPROPYL ALCOHOL) SPRAY LIGHT 6-PACK HAND SANITIZER (ISOPROPYL ALCOHOL) SPRAY [HANDETIZER, LLC]","Aug 19, 2025",1
3,14% Azelaic Acid Acne,4ccaa734-83be-611b-e063-6294a90ab3bf,"14% AZELAIC ACID ACNE (AZELAIC ACID) CREAM [GUANGDONG BISUTANG BIOTECHNOLOGY CO., LTD.]","Mar 13, 2026",1
4,1ST MEDXPATCH WITH LIDOCAINE 4%-RX,b55585cf-45da-4b78-b68e-f7dddf23bfe7,"1ST MEDXPATCH WITH LIDOCAINE 4%-RX (LIDOCAINE, CAPSAICIN, MENTHOL, METHYL SALICYLATE) PATCH [1ST MEDX LLC]","Jan 09, 2026",11


In [18]:
import os

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

with open(f"{OUTPUT_FOLDER}/dailymed_metadata.json", "w") as f:
    json.dump(dailymed_metadata, f, indent=4)

Metadata

In [20]:
dailymed_metadata = {

    "build_timestamp":

        datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "rows":

        int(
            len(
                dailymed_repository_df
            )
        )
}

with open(

    f"{OUTPUT_FOLDER}/dailymed_metadata.json",

    "w"
) as f:

    json.dump(
        dailymed_metadata,
        f,
        indent=4
    )

Drug Container Data

In [21]:
from bs4 import BeautifulSoup

def extract_dailymed_attributes(
    setid
):

    url = (
        "https://dailymed.nlm.nih.gov/"
        f"dailymed/services/v2/spls/{setid}.xml"
    )

    response = requests.get(
        url,
        timeout=60
    )

    if response.status_code != 200:

        return None

    soup = BeautifulSoup(
        response.text,
        "xml"
    )

    full_text = soup.get_text(
        separator=" ",
        strip=True
    )

    return {

        "setid":
            setid,

        "single_dose_vial":
            "single-dose vial"
            in full_text.lower(),

        "multi_dose_vial":
            "multi-dose vial"
            in full_text.lower(),

        "preservative_free":
            "does not contain a preservative"
            in full_text.lower(),

        "refrigerated":
            (
                "2°c to 8°c"
                in full_text.lower()
            ),

        "do_not_freeze":
            (
                "do not freeze"
                in full_text.lower()
            ),

        "do_not_shake":
            (
                "do not shake"
                in full_text.lower()
            )
    }

In [22]:
drug_master_df = pd.read_parquet(
    "master_repository/drug_master.parquet"
)

def drug_lookup(
    search_value
):

    search_value = str(
        search_value
    ).strip().upper()

    results = drug_master_df[

        drug_master_df[
            "brand_name"
        ]
        .fillna("")
        .str.upper()
        .str.contains(
            search_value,
            na=False
        )

        |

        drug_master_df[
            "generic_name"
        ]
        .fillna("")
        .str.upper()
        .str.contains(
            search_value,
            na=False
        )

        |

        drug_master_df[
            "package_ndc"
        ]
        .fillna("")
        .astype(str)
        .str.contains(
            search_value,
            na=False
        )

        |

        drug_master_df[
            "product_ndc"
        ]
        .fillna("")
        .astype(str)
        .str.contains(
            search_value,
            na=False
        )

    ]

    return results

In [23]:
results = drug_lookup(
    "KEYTRUDA"
)

print(
    f"Matches: {len(results):,}"
)

display(
    results.head()
)

Matches: 2


,brand_name,generic_name,ingredient_name,manufacturer,product_ndc,package_ndc,package_description,dosage_form,route,marketing_category,application_number,product_type,rxcui,name,synonym,tty
11435,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,BERAHYALURONIDASE ALFA,Merck Sharp & Dohme LLC,0006-5083,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761467,HUMAN PRESCRIPTION DRUG,None,None,None,None
11436,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,PEMBROLIZUMAB,Merck Sharp & Dohme LLC,0006-5083,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761467,HUMAN PRESCRIPTION DRUG,None,None,None,None


In [24]:
def drug_search(
    search_value
):

    results = drug_lookup(
        search_value
    )

    print(
        f"Matches: {len(results):,}"
    )

    return results

In [25]:
def drug_profile(
    search_value
):

    results = drug_lookup(
        search_value
    )

    if len(results) == 0:

        print(
            "No matches found."
        )

        return

    print(
        f"Matches Found: {len(results)}"
    )

    print()

    for _, record in results.iterrows():

        print("=" * 80)

        for column in record.index:

            print(
                f"{column}: {record[column]}"
            )

        print("=" * 80)

        print()

In [26]:
drug_search(
    "KEYTRUDA"
)

Matches: 2


,brand_name,generic_name,ingredient_name,manufacturer,product_ndc,package_ndc,package_description,dosage_form,route,marketing_category,application_number,product_type,rxcui,name,synonym,tty
11435,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,BERAHYALURONIDASE ALFA,Merck Sharp & Dohme LLC,0006-5083,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761467,HUMAN PRESCRIPTION DRUG,None,None,None,None
11436,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,PEMBROLIZUMAB,Merck Sharp & Dohme LLC,0006-5083,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761467,HUMAN PRESCRIPTION DRUG,None,None,None,None


In [27]:
def get_drug_record(
    search_value
):

    results = drug_lookup(
        search_value
    )

    if len(results) == 0:

        return None

    return (
        results
        .iloc[0]
        .to_dict()
    )

In [28]:
def get_drug_record(
    search_value
):

    results = drug_lookup(
        search_value
    )

    if len(results) == 0:

        return None

    return (
        results
        .iloc[0]
        .to_dict()
    )

In [29]:
from bs4 import BeautifulSoup

def build_dailymed_attributes_repository(
    drug_master_df,
    sample_size=None
):

    rows = []

    drug_names = (

        drug_master_df[
            "brand_name"
        ]

        .dropna()

        .drop_duplicates()
    )

    if sample_size:

        drug_names = (
            drug_names
            .head(sample_size)
        )

    for drug_name in drug_names:

        try:

            response = requests.get(

                "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json",

                params={
                    "drug_name":
                        drug_name
                },

                timeout=60
            )

            data = response.json()

            spls = data.get(
                "data",
                []
            )

            if len(spls) == 0:

                continue

            spl = spls[0]

            setid = spl[
                "setid"
            ]

            xml_response = requests.get(

                f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{setid}.xml",

                timeout=60
            )

            soup = BeautifulSoup(

                xml_response.text,

                "xml"
            )

            full_text = soup.get_text(

                separator=" ",

                strip=True
            ).lower()

            rows.append({

                "brand_name":
                    drug_name,

                "setid":
                    setid,

                "single_dose_vial":
                    "single-dose vial"
                    in full_text,

                "multi_dose_vial":
                    "multi-dose vial"
                    in full_text,

                "preservative_free":
                    "does not contain a preservative"
                    in full_text,

                "refrigerated":
                    (
                        "2°c to 8°c"
                        in full_text
                    ),

                "do_not_freeze":
                    (
                        "do not freeze"
                        in full_text
                    ),

                "do_not_shake":
                    (
                        "do not shake"
                        in full_text
                    )
            })

        except Exception:

            continue

    return pd.DataFrame(
        rows
    )

In [30]:
from bs4 import BeautifulSoup

def build_dailymed_attributes_repository(
    drug_master_df,
    sample_size=10
):

    rows = []

    drug_names = (

        drug_master_df[
            "brand_name"
        ]

        .dropna()

        .drop_duplicates()

        .head(sample_size)

        .tolist()
    )

    for i, drug_name in enumerate(
        drug_names,
        start=1
    ):

        print(
            f"{i}/{len(drug_names)} - {drug_name}"
        )

        try:

            response = requests.get(

                "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json",

                params={
                    "drug_name": drug_name
                },

                timeout=10
            )

            spls = response.json().get(
                "data",
                []
            )

            print(
                f"  SPLs Found: {len(spls)}"
            )

            if len(spls) == 0:

                continue

            setid = spls[0]["setid"]

            xml_response = requests.get(

                f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{setid}.xml",

                timeout=10
            )

            full_text = xml_response.text.lower()

            rows.append({

                "brand_name": drug_name,

                "setid": setid,

                "single_dose_vial":
                    "single-dose vial"
                    in full_text,

                "multi_dose_vial":
                    "multi-dose vial"
                    in full_text,

                "preservative_free":
                    "does not contain a preservative"
                    in full_text,

                "refrigerated":
                    (
                        "2°c to 8°c"
                        in full_text
                    ),

                "do_not_freeze":
                    (
                        "do not freeze"
                        in full_text
                    ),

                "do_not_shake":
                    (
                        "do not shake"
                        in full_text
                    )
            })

        except Exception as e:

            print(
                f"  ERROR: {e}"
            )

    return pd.DataFrame(
        rows
    )

In [31]:
dailymed_attributes_df = (
    build_dailymed_attributes_repository(
        drug_master_df,
        sample_size=5
    )
)

1/5 - TRIDERMA EVERYDAY BRUISE RELIEF
  SPLs Found: 1
2/5 - Oxy Advanced Care Rapid Spot Treatment
  SPLs Found: 2
3/5 - medicated pads
  SPLs Found: 31
4/5 - Celecoxib
  SPLs Found: 100
5/5 - TRIAMCINOLONE ACETONIDE
  SPLs Found: 100


In [32]:
import json

with open(f"{OUTPUT_FOLDER}/dailymed_metadata.json", "r") as f:
    metadata = json.load(f)

print(metadata)

{'build_timestamp': '2026-07-02 09:15:47', 'rows': 120884}


In [33]:
from pathlib import Path

metadata_file = Path(OUTPUT_FOLDER) / "dailymed_metadata.json"
print(metadata_file.resolve())
print(metadata_file.exists())

C:\Users\barry.peterson\OneDrive - Integra Connect LLC\Documents\Pharmacy Reports\Master Drug File\dailymed_repository\dailymed_metadata.json
True
